In [1]:
# Keep repository-relative paths valid from notebook subfolders.
from pathlib import Path
import os

os.chdir(next(
    root for root in (Path.cwd(), *Path.cwd().parents)
    if (root / "notebooks").is_dir() and (root / "requirements.txt").is_file()
))

from pathlib import Path
import pandas as pd
import re

In [2]:
DATA_ROOT = Path("data/idx_financial/raw")

print("Dataset path:")
print(DATA_ROOT.resolve())

print("\nExists:")
print(DATA_ROOT.exists())

Dataset path:
E:\BINUS_CODING\Ai Builders Hackhaton\AI-Builders-Hackhaton-2026-Backend\data\idx_financial_statements

Exists:
True


In [3]:
files = []

for file_path in DATA_ROOT.rglob("*"):
    if not file_path.is_file():
        continue

    files.append(
        {
            "file_path": str(file_path),
            "file_name": file_path.name,
            "extension": file_path.suffix.lower(),
            "size_mb": round(file_path.stat().st_size / (1024 * 1024), 3)
        }
    )

inventory_df = pd.DataFrame(files)

print("Total files:", len(inventory_df))

display(inventory_df.head(20))

Total files: 114366


,file_path,file_name,extension,size_mb
0,data\idx_financial_statements\Financial_Statem...,AADI.zip,.zip,4.045
1,data\idx_financial_statements\Financial_Statem...,BBRI.zip,.zip,8.324
2,data\idx_financial_statements\Financial_Statem...,ZYRX_2025_Q1_FinancialStatement-2025-I-ZYRX.pdf,.pdf,0.805
3,data\idx_financial_statements\Financial_Statem...,ZYRX_2025_Q1_FS.xlsx,.xlsx,0.338
4,data\idx_financial_statements\Financial_Statem...,ZYRX_2025_Q1_Instance.zip,.zip,0.123
5,data\idx_financial_statements\Financial_Statem...,ZYRX_2025_Q1_LK PT Zyrexindo Mandiri Buana Tbk...,.pdf,1.055
6,data\idx_financial_statements\Financial_Statem...,ZYRX_2025_Q1_SPD Maret 2025.pdf,.pdf,0.340
7,data\idx_financial_statements\Financial_Statem...,ZYRX_2025_Q1_XBRL.zip,.zip,0.249
8,data\idx_financial_statements\Financial_Statem...,ZONE_2025_Q1_2025 - Checklist Laporan Keuangan...,.pdf,1.062
9,data\idx_financial_statements\Financial_Statem...,ZONE_2025_Q1_2025 - KI Kenaikan Penurunan Q1.pdf,.pdf,1.280


In [4]:
total_size_mb = (inventory_df["size_mb"].sum())

total_size_gb = (total_size_mb / 1024)

print("Total size:", round(total_size_gb, 2), "GB")

Total size: 103.99 GB


In [5]:
extension_summary = (
    inventory_df
    .groupby("extension")
    .agg(
        file_count=("file_name", "count"),
        total_size_mb=("size_mb", "sum")
    )
    .reset_index()
    .sort_values(
        "file_count",
        ascending=False
    )
)

extension_summary["total_size_gb"] = (extension_summary["total_size_mb"] / 1024)

display(extension_summary)

,extension,file_count,total_size_mb,total_size_gb
1,.pdf,66140,94964.230,92.738506
3,.zip,32048,3539.573,3.456614
2,.xlsx,16003,6137.728,5.993875
0,.crdownload,175,1842.067,1.798894


In [6]:
def extract_year(file_path):

    text = str(file_path)

    match = re.search(
        r"Financial_Statement_(20\d{2})",
        text,
        flags=re.IGNORECASE
    )

    if match:
        return int(
            match.group(1)
        )

    return None

inventory_df["year"] = (
    inventory_df["file_path"]
    .apply(extract_year)
)

display(
    inventory_df[
        [
            "file_name",
            "extension",
            "year"
        ]
    ].head(20)
)

,file_name,extension,year
0,AADI.zip,.zip,2025
1,BBRI.zip,.zip,2025
2,ZYRX_2025_Q1_FinancialStatement-2025-I-ZYRX.pdf,.pdf,2025
3,ZYRX_2025_Q1_FS.xlsx,.xlsx,2025
4,ZYRX_2025_Q1_Instance.zip,.zip,2025
5,ZYRX_2025_Q1_LK PT Zyrexindo Mandiri Buana Tbk...,.pdf,2025
6,ZYRX_2025_Q1_SPD Maret 2025.pdf,.pdf,2025
7,ZYRX_2025_Q1_XBRL.zip,.zip,2025
8,ZONE_2025_Q1_2025 - Checklist Laporan Keuangan...,.pdf,2025
9,ZONE_2025_Q1_2025 - KI Kenaikan Penurunan Q1.pdf,.pdf,2025


In [7]:
year_summary = (
    inventory_df
    .groupby("year", dropna=False)
    .size()
    .reset_index(
        name="file_count"
    )
)

display(
    year_summary
)

,year,file_count
0,2020,18063
1,2021,19776
2,2022,22035
3,2023,24509
4,2024,24579
5,2025,5404


In [8]:
def extract_quarter(file_path):

    text = str(file_path).upper()

    patterns = [
        r"\bQ([1-4])\b",
        r"QUARTER[_\-\s]?([1-4])",
        r"QTR[_\-\s]?([1-4])"
    ]

    for pattern in patterns:

        match = re.search(
            pattern,
            text
        )

        if match:
            return (
                "Q"
                + match.group(1)
            )

    return None

inventory_df["quarter"] = (
    inventory_df["file_path"]
    .apply(extract_quarter)
)

display(
    inventory_df[
        [
            "file_name",
            "year",
            "quarter",
            "extension"
        ]
    ].head(30)
)

,file_name,year,quarter,extension
0,AADI.zip,2025,NaN,.zip
1,BBRI.zip,2025,NaN,.zip
2,ZYRX_2025_Q1_FinancialStatement-2025-I-ZYRX.pdf,2025,Q1,.pdf
3,ZYRX_2025_Q1_FS.xlsx,2025,Q1,.xlsx
4,ZYRX_2025_Q1_Instance.zip,2025,Q1,.zip
5,ZYRX_2025_Q1_LK PT Zyrexindo Mandiri Buana Tbk...,2025,Q1,.pdf
6,ZYRX_2025_Q1_SPD Maret 2025.pdf,2025,Q1,.pdf
7,ZYRX_2025_Q1_XBRL.zip,2025,Q1,.zip
8,ZONE_2025_Q1_2025 - Checklist Laporan Keuangan...,2025,Q1,.pdf
9,ZONE_2025_Q1_2025 - KI Kenaikan Penurunan Q1.pdf,2025,Q1,.pdf


In [9]:
quarter_summary = (
    inventory_df
    .groupby(
        "quarter",
        dropna=False
    )
    .size()
    .reset_index(
        name="file_count"
    )
)

display(
    quarter_summary
)

,quarter,file_count
0,Q1,28184
1,Q2,26755
2,Q3,24784
3,Q4,34641
4,NaN,2


In [10]:
unknown_quarter = (
    inventory_df[
        inventory_df[
            "quarter"
        ].isna()
    ]
)

display(
    unknown_quarter[
        [
            "file_path",
            "file_name",
            "year"
        ]
    ].head(50)
)

,file_path,file_name,year
0,data\idx_financial_statements\Financial_Statem...,AADI.zip,2025
1,data\idx_financial_statements\Financial_Statem...,BBRI.zip,2025


In [11]:
sample_paths = (
    inventory_df[
        "file_path"
    ]
    .drop_duplicates()
    .head(100)
)

for path in sample_paths:
    print(path)

data\idx_financial_statements\Financial_Statement_2025_Q2\AADI.zip
data\idx_financial_statements\Financial_Statement_2025_Q2\BBRI.zip
data\idx_financial_statements\Financial_Statement_2025_Q2\ZYRX\2025\Q1\ZYRX_2025_Q1_FinancialStatement-2025-I-ZYRX.pdf
data\idx_financial_statements\Financial_Statement_2025_Q2\ZYRX\2025\Q1\ZYRX_2025_Q1_FS.xlsx
data\idx_financial_statements\Financial_Statement_2025_Q2\ZYRX\2025\Q1\ZYRX_2025_Q1_Instance.zip
data\idx_financial_statements\Financial_Statement_2025_Q2\ZYRX\2025\Q1\ZYRX_2025_Q1_LK PT Zyrexindo Mandiri Buana Tbk Mar 2025.pdf
data\idx_financial_statements\Financial_Statement_2025_Q2\ZYRX\2025\Q1\ZYRX_2025_Q1_SPD Maret 2025.pdf
data\idx_financial_statements\Financial_Statement_2025_Q2\ZYRX\2025\Q1\ZYRX_2025_Q1_XBRL.zip
data\idx_financial_statements\Financial_Statement_2025_Q2\ZONE\2025\Q1\ZONE_2025_Q1_2025 - Checklist Laporan Keuangan MP Q1.pdf
data\idx_financial_statements\Financial_Statement_2025_Q2\ZONE\2025\Q1\ZONE_2025_Q1_2025 - KI Kenaikan 

In [12]:
xlsx_files = (
    inventory_df[
        inventory_df[
            "extension"
        ].isin(
            [".xlsx", ".xls"]
        )
    ]
)

pdf_files = (
    inventory_df[
        inventory_df[
            "extension"
        ] == ".pdf"
    ]
)

zip_files = (
    inventory_df[
        inventory_df[
            "extension"
        ] == ".zip"
    ]
)

print(
    "Excel files:",
    len(xlsx_files)
)

print(
    "PDF files:",
    len(pdf_files)
)

print(
    "ZIP files:",
    len(zip_files)
)

Excel files: 16003
PDF files: 66140
ZIP files: 32048


In [13]:
coverage_table = (
    inventory_df
    .pivot_table(
        index="year",
        columns="extension",
        values="file_name",
        aggfunc="count",
        fill_value=0
    )
)

display(
    coverage_table
)

extension,.crdownload,.pdf,.xlsx,.zip
year,,,,
2020,0,10006,2690,5367
2021,4,11105,2869,5798
2022,27,12709,3121,6178
2023,105,14639,3229,6536
2024,36,14773,3280,6490
2025,3,2908,814,1679


In [14]:
OUTPUT_FILE = Path(
    "data/idx_financial/inventory/idx_dataset_inventory.csv"
)

inventory_df.to_csv(
    OUTPUT_FILE,
    index=False
)

print(
    "Saved to:",
    OUTPUT_FILE.resolve()
)

Saved to: E:\BINUS_CODING\Ai Builders Hackhaton\AI-Builders-Hackhaton-2026-Backend\data\idx_dataset_inventory.csv
